# មេរៀនទី ១១ - ពិធីការបរិបទគំរូ (MCP)

**ពិធីការបរិបទគំរូ (MCP)** ជាស្ដង់ដារបើកដែលអនុញ្ញាតឱ្យភ្នាក់ងារស្វែងរក និងប្រើឧបករណ៍ ប្រភពធនធាន និងប្រភពទិន្នន័យដោយអាស្រ័យលើruntime។ ជំនួសការរួមបញ្ចូលឧបករណ៍ជាក់លាក់ទៅក្នុងភ្នាក់ងារ MCP អនុញ្ញាតឱ្យភ្នាក់ងារតភ្ជាប់ទៅម៉ាស៊ីនបម្រើខាងក្រៅដែលបង្ហាញសមត្ថភាពតាមតម្រូវការ។

ក្នុងមេរៀននេះ អ្នកនឹងរៀន៖
- MCP គឺជាអ្វី និងហេតុអ្វីវាមានសារៈសំខាន់ចំពោះប្រព័ន្ធភ្នាក់ងារ
- របៀបស្ថាបត្យកម្មម៉ាស៊ីនបម្រើ-គ្លាយហ្វ្ញេ MCP ធ្វើការយ៉ាងដូចម្តេច
- របៀបបង្កើតភ្នាក់ងារដែលប្រើការស្វែងរកឧបករណ៍បែប MCP


## ការតំឡើង

**លក្ខខណ្ឌមុន:**
- គំរោង Microsoft Foundry ដែលមានម៉ូដែលបានប្រាប់ចេញ
- រត់ `az login` សម្រាប់ការផ្ទៀងផ្ទាត់


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from typing import Annotated
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## តើពិធីការបរិបទម៉ូដែល (MCP) គឺជាអ្វី?

MCP កំណត់វិធីស្តង់ដាដើម្បីឱ្យភ្នាក់ងារបញ្ញាសិប្បនិម្មិតស្វែងរក និងមានអន្តរកម្មជាមួយឧបករណ៍ និងប្រភពទិន្នន័យខាងក្រៅ:

- **ម៉ាស៊ែរម៉ូដែល MCP**: បង្ហាញឧបករណ៍, ប្រភពធនធាន, និងការកំណត់មុនតាមរយៈពិធីការស្តង់ដារ
- **អតិថិជន MCP**: ជារយៈពេលធ្វើការរបស់ភ្នាក់ងារ ដែលភ្ជាប់ទៅម៉ាស៊ែរនិងស្វែងរកសមត្ថភាពដែលមាន
- **ការស្វែងរក δυναμική**: ភ្នាក់ងារមិនត្រូវការឧបករណ៍កូដរឹង — ពួកគេចាប់ផ្តើមស្វែងរកអ្វីដែលមាននៅពេលដំណើរការ

វានេះមានអំណាចសម្រាប់ការសាងសង់ប្រព័ន្ធភ្នាក់ងារដែលអាចពង្រីកបាន ដែលសមត្ថភាពថ្មីៗអាចបន្ថែមបានដោយមិនបង្កើតកូដភ្នាក់ងារឡើងវិញ។


## របៀបដំណើរការ MCP

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. ដៃគូចត្រូវបានភ្ជាប់ជាមួយម៉ាស៊ីនមេ MCP
2. ម៉ាស៊ីនមេឆ្លើយតបជាមួយបញ្ជីឧបករណ៍ដែលអាចប្រើបាន និងរចនាសម្ព័ន្ធរបស់វា
3. ដៃគូច្ាចហៅឧបករណ៍ណាមួយដែលបានរកឃើញនៅពេលវាពិចារណា
4. លទ្ធផលត្រឡប់មកវិញតាមរយៈពិធីការដដែល


## ការសម្រួលការស្វែងរកឧបករណ៍ MCP  

ពីព្រោះម៉ាស៊ីនមេ MCP ពិតប្រាកដត្រូវការប្រតិបត្តិការ ម៉ាស៊ីនមេដំណើរការ យើងនឹងបង្ហាញរូបមន្តដោយប្រើមុខងារ `@tool` ដែលសម្រួលនូវអ្វីដែលសេវាកម្មស្នាក់នៅជាប់ MCP នឹងផ្តល់ជូន។  

ក្នុងការផលិត ឧបករណ៍ទាំងនេះនឹងត្រូវបានស្វែងរកដោយឌើងមែនពីម៉ាស៊ីនមេ MCP មិនមែនកំណត់ក្នុងតំបន់ដែនទេ។  


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## ការបង្កើតភ្នាក់ងារជាមួយឧបករណ៍បែប MCP


In [ ]:
agent = client.as_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP ក្នុងដំណើរការផលិត

នៅក្នុងបរិយាកាសការផលិត MCP អនុញ្ញាតឲ្យមានលំនាំដ៏មានឥណទានខ្លាំង:

- **ការស្វែងរកឧបករណ៍ឌីណាមិក**: អ្នកភ្នាក់ងារតភ្ជាប់ទៅម៉ាស៊ីនបម្រើ MCP និងស្វែងរកឧបករណ៍នៅពេលរត់
- **រចនាសម្ព័ន្ធបំបែកដោយឯករាជ្យ**: អ្នកផ្តល់ឧបករណ៍អាចធ្វើការអាប់ដេតដោយឯករាជ្យពីអ្នកភ្នាក់ងារ
- **ការចែករំលែកអន្តរក្រុមក្រុមហ៊ុន**: ក្រុមការងារអាចបង្ហាញសមត្ថភាពតាមរយៈម៉ាស៊ីនបម្រើ MCP ដែលអ្នកភ្នាក់ងារណាមួយអាចប្រើបាន
- **ការគាំទ្រដោយ Microsoft Agent Framework**: MAF មានការគាំទ្រម៉ូឌុល MCP លំហាត់ក្នុងតាមរយៈការតភ្ជាប់ `mcp`

ដើម្បីប្រើម៉ាស៊ីនបម្រើ MCP អ្នកពិតជាមួយ MAF អ្នកនឹងត្រូវតភ្ជាប់តាមរយៈ `hosted_mcp_tool()` ឬការតភ្ជាប់ម៉ូឌុល MCP client។

**សូមរៀនបន្ថែម:**
- [MCP Specification](https://modelcontextprotocol.io/)
- [Microsoft Agent Framework MCP Support](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## សង្ខេប

ក្នុងមេរៀននេះ អ្នកបានរៀនពី៖
- **MCP** គឺជាស្ដង់ដានៅលើភоскសម្រាប់ការស្វែងរកឧបករណ៍ឲ្យអត្តៈធាតុ និងអ្នកផ្គត់ផ្គង់ឧបករណ៍ដោយ δυναμικός
- រចនាសម្ព័ន្ធ **client-server** អនុញ្ញាតឲ្យអត្តៈធាតុស្វែងរកសមត្ថភាពនៅពេលរត់
- MCP អនុញ្ញាតឲ្យមាន **ប្រព័ន្ធអត្តៈធាតុប្រើប្រាស់បំបែក និងពង្រីក** ដែលឧបករណ៍អាចត្រូវបានបន្ថែមដោយមិនដូរលេខកូដ
- Microsoft Agent Framework ផ្ដល់នូវ **ការ​គាំទ្រ MCP ក្នុងសំណុំបែបបទ built-in** សម្រាប់ការប្រើប្រាស់ក្នុងការផលិត


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការបដិសេធ**:
ឯកសារនេះត្រូវបានបម្លែងភាសា ដោយប្រើសេវាបម្លែងភាសា AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ទោះយើងខ្ញុំមានក្តីប្រាថ្នាឱ្យបានច្បាស់លាស់ តែសូមយល់ដឹងថាការបម្លែងដោយស្វ័យប្រវត្តិក៏អាចមានកំហុសឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមជាភាសាទីតាំងគួរត្រូវបានគេប្រើជាប្រភពច្បាស់លាស់។ សម្រាប់ព័ត៌មានសំខាន់ៗ សូមណែនាំឱ្យប្រើប្រាស់ការប្រែដោយមនុស្សជំនាញ។ យើងខ្ញុំមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសបន្ទាប់ពីការប្រើប្រាស់ការបម្លែងនេះនោះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
